# Cross-Model Comparison

Aggregates the 12 `SweepResult` pickles from `data/output/seed_sweep_results/` into a leaderboard, per-class breakdowns, paired significance tests, and a consensus error matrix.

**Primary sort: MCC** (Matthews correlation coefficient). F1 macro, minority F1 macro, balanced accuracy, and accuracy shown alongside.

Hard-fails if any of the 12 expected pickles is missing — rerun the corresponding model's sweep first.

In [1]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import SWEEP_DIR, FIG_DIR, save_plot
from utils.constants import CLASS_NAMES, COLORS

## 1. Setup & load

Load every expected pickle, build a long-form dataframe of (model, seed) → metrics, and confirm every model has the same seed count.

In [2]:
EXPECTED_MODELS = [
    "dtree", "extrees", "gnb", "knn", "lda", "logreg",
    "mlp", "qda", "rbfsvm", "rforest", "svm", "xgb",
]

PRETTY_NAMES = {
    "dtree":   "Decision Tree",
    "extrees": "Extra Trees",
    "rforest": "Random Forest",
    "xgb":     "XGBoost",
    "mlp":     "MLP",
    "svm":     "Linear SVM",
    "rbfsvm":  "RBF SVM",
    "logreg":  "Logistic Regression",
    "lda":     "LDA",
    "qda":     "QDA",
    "knn":     "KNN",
    "gnb":     "Gaussian NB",
}

missing = [m for m in EXPECTED_MODELS if not (SWEEP_DIR / f"sweep_{m}.pkl").exists()]
if missing:
    raise FileNotFoundError(
        f"Missing sweep pickles for {len(missing)} model(s): {missing}. "
        f"Rerun the relevant sweep(s) before regenerating the comparison."
    )

results = {}
for m in EXPECTED_MODELS:
    with open(SWEEP_DIR / f"sweep_{m}.pkl", "rb") as f:
        results[PRETTY_NAMES[m]] = pickle.load(f)

long_df = pd.concat(
    [r.df.assign(model=name) for name, r in results.items()],
    ignore_index=True,
)

seed_counts = long_df.groupby("model")["seed"].nunique()
assert seed_counts.nunique() == 1, f"Seed-count mismatch across models:\n{seed_counts}"
N_SEEDS = int(seed_counts.iloc[0])

print(f"Loaded {len(results)} models \u00d7 {N_SEEDS} seeds = {len(long_df)} rows")

Loaded 12 models × 50 seeds = 600 rows


## 2. Master leaderboard

Sorted by mean MCC. All cells are `mean ± std` across the `N_SEEDS` random splits. **Bold** marks the per-column winner.

In [6]:
LEADERBOARD_COLS = [
    ("mcc",               "MCC"),
    ("f1_macro",          "F1 macro"),
    ("minority_f1_macro", "Minority F1 macro"),
    ("balanced_acc",      "Balanced acc"),
    ("accuracy",          "Accuracy"),
]

agg = long_df.groupby("model").agg(
    {col: ["mean", "std"] for col, _ in LEADERBOARD_COLS}
)

display_df = pd.DataFrame(index=agg.index)
for col, label in LEADERBOARD_COLS:
    display_df[label] = (
        agg[(col, "mean")].map("{:.4f}".format)
        + " ± "
        + agg[(col, "std")].map("{:.4f}".format)
    )

display_df["_sort"] = agg[("mcc", "mean")]
display_df = display_df.sort_values("_sort", ascending=False).drop(columns="_sort")
display_df.index.name = "Model"

winners = {
    label: (agg[(col, "mean")].idxmax(), agg[(col, "mean")].max())
    for col, label in LEADERBOARD_COLS
}

print("── Per-column winners ──")
width = max(len(label) for label in winners) + 1
for label, (model, value) in winners.items():
    print(f"  {label:<{width}} {model}  ({value:.4f})")
print()

display_df

── Per-column winners ──
  MCC                XGBoost  (0.6535)
  F1 macro           XGBoost  (0.7512)
  Minority F1 macro  XGBoost  (0.6304)
  Balanced acc       XGBoost  (0.9283)
  Accuracy           XGBoost  (0.9853)



,MCC,F1 macro,Minority F1 macro,Balanced acc,Accuracy
Model,,,,,
XGBoost,0.6535 ± 0.0596,0.7512 ± 0.0509,0.6304 ± 0.0757,0.9283 ± 0.0545,0.9853 ± 0.0040
Random Forest,0.6465 ± 0.0545,0.7399 ± 0.0469,0.6134 ± 0.0699,0.9246 ± 0.0534,0.9850 ± 0.0034
Decision Tree,0.6142 ± 0.0813,0.7229 ± 0.0649,0.5887 ± 0.0962,0.9172 ± 0.0697,0.9821 ± 0.0066
Extra Trees,0.5433 ± 0.0555,0.6499 ± 0.0545,0.4804 ± 0.0809,0.8748 ± 0.0634,0.9767 ± 0.0049
Linear SVM,0.4753 ± 0.0631,0.6009 ± 0.0517,0.4096 ± 0.0762,0.8724 ± 0.0737,0.9670 ± 0.0078
Gaussian NB,0.4445 ± 0.0404,0.5694 ± 0.0403,0.3635 ± 0.0595,0.8576 ± 0.0655,0.9616 ± 0.0061
RBF SVM,0.4149 ± 0.0531,0.5822 ± 0.0496,0.3822 ± 0.0734,0.8146 ± 0.0736,0.9642 ± 0.0063
QDA,0.4056 ± 0.0550,0.5833 ± 0.0471,0.3858 ± 0.0695,0.8297 ± 0.0726,0.9569 ± 0.0072
Logistic Regression,0.3907 ± 0.0387,0.5176 ± 0.0362,0.2900 ± 0.0528,0.8743 ± 0.0674,0.9456 ± 0.0082
